# 🔬 Análise Exploratória — Exames Citopatológicos Alterados (10ª CRS)

EDA sobre `tcc-analise-siscan-10crs.gold.fato_exames_alterados` (município × ano × faixa etária, 2020–2025), construída sobre a gold reprocessada em 03/nov/2026 — o denominador agora exclui amostras insatisfatórias (390) e rejeitadas (70), que não geram laudo diagnóstico definitivo, restando 130.598 exames analisáveis; 4.479 alterados.

**Perguntas norteadoras:**
1. Qual faixa etária concentra a maior taxa de exames alterados?
2. A taxa mudou ao longo do tempo (2020–2025)?
3. Quais municípios se destacam — e qual o efeito de bases pequenas (IC 95%)?
4. Existe relação entre volume de exames e taxa de alteração?
5. É possível “prever” algo? → modelo logístico binomial (prob. de alteração por faixa/ano) e projeção de tendência, com limitações explícitas.

**Regra metodológica:** taxas sempre pela **razão de somas** (Σ alterados ÷ Σ válidos), nunca média de percentuais; faixas `Ignorado` e `Até 9 anos` excluídas das taxas (base ínfima — mesma regra do dashboard AI/BI).

In [0]:
# ---------------------------------------------------------------------------
# 1. CARGA E VISÃO GERAL
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import functions as F

df = spark.table("`tcc-analise-siscan-10crs`.gold.fato_exames_alterados").cache()

n_rows = df.count()
tot = df.select(
    F.sum("total_exames_validos").alias("validos"),
    F.sum("quantidade_alterados").alias("alterados"),
).collect()[0]

print(f"Linhas (município × ano × faixa): {n_rows}")
print(f"Total de exames válidos: {tot['validos']:,}")
print(f"Total de exames alterados: {tot['alterados']:,}")
print(f"Taxa global ponderada: {tot['alterados'] / tot['validos'] * 100:.3f}%")
print("\nCobertura e volume por ano:")
df.select(
    F.countDistinct("municipio").alias("municipios"),
    F.countDistinct("ano").alias("anos"),
    F.countDistinct("faixa_etaria").alias("faixas"),
).show()
df.groupBy("ano").agg(
    F.sum("total_exames_validos").alias("validos"),
    F.sum("quantidade_alterados").alias("alterados"),
    F.round(F.sum("quantidade_alterados") / F.sum("total_exames_validos") * 100, 2).alias("taxa_pct"),
).orderBy("ano").show()

In [0]:
# ---------------------------------------------------------------------------
# 2. TAXA PONDERADA POR FAIXA ETÁRIA (razão de somas — nunca média de %)
# ---------------------------------------------------------------------------
FAIXAS_EXCLUIR = ["Ignorado", "Até 9 anos"]  # base ínfima / fora da diretriz

taxa_faixa = (
    df.filter(~F.col("faixa_etaria").isin(FAIXAS_EXCLUIR))
    .groupBy("faixa_etaria")
    .agg(
        F.sum("quantidade_alterados").alias("alterados"),
        F.sum("total_exames_validos").alias("validos"),
    )
    .withColumn("taxa_pct", F.round(F.col("alterados") / F.col("validos") * 100, 2))
    .orderBy(F.desc("taxa_pct"))
)
display(taxa_faixa)

tf = taxa_faixa.toPandas()
top = tf.iloc[0]
print(f">>> Maior taxa de alteração: {top['faixa_etaria']} — {top['taxa_pct']}% "
      f"({top['alterados']:.0f} alterados de {top['validos']:.0f} exames)")

In [0]:
# ---------------------------------------------------------------------------
# 3. HEATMAP — TAXA (%) POR FAIXA × ANO
# ---------------------------------------------------------------------------
pivot = (
    df.filter(~F.col("faixa_etaria").isin(FAIXAS_EXCLUIR))
    .groupBy("faixa_etaria", "ano")
    .agg(F.round(F.sum("quantidade_alterados") / F.sum("total_exames_validos") * 100, 2).alias("taxa"))
    .toPandas()
    .pivot(index="faixa_etaria", columns="ano", values="taxa")
)

ordem = ["10 a 14 anos", "15 a 19 anos", "20 a 24 anos", "25 a 29 anos", "30 a 34 anos",
         "35 a 39 anos", "40 a 44 anos", "45 a 49 anos", "50 a 54 anos", "55 a 59 anos",
         "60 a 64 anos", "65 a 69 anos", "70 a 74 anos", "75 a 79 anos", "Acima de 79 anos"]
pivot = pivot.reindex([f for f in ordem if f in pivot.index])

fig, ax = plt.subplots(figsize=(11, 6))
im = ax.imshow(pivot.values.astype(float), cmap="OrRd", aspect="auto")
ax.set_xticks(range(len(pivot.columns)), [str(int(c)) for c in pivot.columns])
ax.set_yticks(range(len(pivot.index)), pivot.index)
ax.tick_params(labelsize=8)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, label="Taxa de alterados (%)")
ax.set_title("Taxa de exames alterados por faixa etária e ano — 10ª CRS (2020–2025)")
plt.tight_layout()
plt.show()

In [0]:
# ---------------------------------------------------------------------------
# 4. EVOLUÇÃO ANUAL DA TAXA PONDERADA (tendência linear simples)
# ---------------------------------------------------------------------------
anual = (
    df.groupBy("ano")
    .agg(F.sum("quantidade_alterados").alias("alt"), F.sum("total_exames_validos").alias("val"))
    .orderBy("ano")
    .toPandas()
)
anual["taxa"] = anual.alt / anual.val * 100

b1, b0 = np.polyfit(anual.ano, anual.taxa, 1)  # MQO sobre 6 pontos anuais
print(anual[["ano", "alt", "val", "taxa"]].round(3).to_string(index=False))
print(f"\nTendência linear: {b1:+.3f} pontos percentuais por ano")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(anual.ano, anual.taxa, marker="o", label="Taxa observada")
ax.plot(anual.ano, b0 + b1 * anual.ano, "--", label="Tendência linear")
for _, r in anual.iterrows():
    ax.annotate(f"{r.taxa:.2f}%", (r.ano, r.taxa), textcoords="offset points", xytext=(0, 7), fontsize=8)
ax.set_xlabel("Ano"); ax.set_ylabel("Taxa (%)")
ax.set_title("Taxa anual de exames alterados — 10ª CRS")
ax.legend()
plt.tight_layout(); plt.show()

In [0]:
# ---------------------------------------------------------------------------
# 5. MUNICÍPIOS — TAXA PONDERADA, IC 95% DE WILSON E CORRELAÇÃO VOLUME × TAXA
# ---------------------------------------------------------------------------
mun = (
    df.filter(~F.col("faixa_etaria").isin(FAIXAS_EXCLUIR))
    .groupBy("municipio")
    .agg(F.sum("quantidade_alterados").alias("alt"), F.sum("total_exames_validos").alias("val"))
    .toPandas()
)
mun["taxa"] = mun.alt / mun.val * 100

def wilson(k, n, z=1.96):
    """Intervalo de confiança 95% para proporção — apropriado para bases pequenas."""
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (c - h) * 100, (c + h) * 100

mun[["ic_lo", "ic_hi"]] = mun.apply(lambda r: pd.Series(wilson(r["alt"], r["val"])), axis=1)
mun = mun.sort_values("taxa", ascending=False).reset_index(drop=True)
print(mun.round(2).to_string(index=False))

# Correlação de Spearman: municípios com mais exames têm taxa maior/menor?
rk_v, rk_t = mun["val"].rank(), mun["taxa"].rank()
rho = np.corrcoef(rk_v, rk_t)[0, 1]
print(f"\nSpearman volume × taxa (municípios): rho = {rho:+.2f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(mun.taxa, range(len(mun)),
            xerr=[mun.taxa - mun.ic_lo, mun.ic_hi - mun.taxa],
            fmt="o", capsize=3)
ax.set_yticks(range(len(mun)), mun.municipio, fontsize=8)
ax.axvline(tot["alterados"] / tot["validos"] * 100, color="gray", ls=":",
           label=f"Taxa regional {tot['alterados'] / tot['validos'] * 100:.2f}%")
ax.set_xlabel("Taxa (%) com IC 95% de Wilson")
ax.set_title("Taxa de alterados por município (2020–2025)")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [0]:
# ---------------------------------------------------------------------------
# 6. POPULAÇÃO-ALVO (25–64) vs SOBRE/SUB-RASTREAMENTO — TAXAS PONDERADAS
# ---------------------------------------------------------------------------
def grupo(faixa: str) -> str:
    if any(faixa.startswith(p) for p in ["25 a 29", "30 a 34", "35 a 39", "40 a 44",
                                          "45 a 49", "50 a 54", "55 a 59", "60 a 64"]):
        return "25–64 (população-alvo)"
    if faixa.startswith(("10 a 14", "15 a 19", "20 a 24")):
        return "<25 (sobrerastreamento)"
    return "65+ (sub-rastreamento)"

g = df.filter(~F.col("faixa_etaria").isin(FAIXAS_EXCLUIR)).toPandas()
g["grupo"] = g["faixa_etaria"].map(grupo)
res = g.groupby("grupo").agg(alt=("quantidade_alterados", "sum"),
                              val=("total_exames_validos", "sum"))
res["taxa_pct"] = (res.alt / res.val * 100).round(2)
print(res.sort_values("taxa_pct", ascending=False).to_string())
print(f"\nSoma das taxas ≠ 100% pois a taxa é de ALTERADOS (não a distribuição de exames).")

In [0]:
# ---------------------------------------------------------------------------
# 7. MODELO LOGÍSTICO BINOMIAL — P(exame alterado) ~ idade + ano
# ---------------------------------------------------------------------------
# Dados agrupados (binomial): cada linha = município×ano×faixa com nº de provas (validos)
# e nº de sucessos (alterados). IRLS implementado em numpy — sem dependências externas
# (scikit-learn/statsmodels não garantidos no serverless).

IDADE = {"10 a 14 anos": 12.5, "15 a 19 anos": 17.5, "20 a 24 anos": 22.5, "25 a 29 anos": 27.5,
         "30 a 34 anos": 32.5, "35 a 39 anos": 37.5, "40 a 44 anos": 42.5, "45 a 49 anos": 47.5,
         "50 a 54 anos": 52.5, "55 a 59 anos": 57.5, "60 a 64 anos": 62.5, "65 a 69 anos": 67.5,
         "70 a 74 anos": 72.5, "75 a 79 anos": 77.5, "Acima de 79 anos": 82.5}

m = df.filter(~F.col("faixa_etaria").isin(FAIXAS_EXCLUIR)).toPandas()
m["idade"] = m["faixa_etaria"].map(IDADE)
m = m.dropna(subset=["idade"]).copy()
# Linhas com denominador ausente/zero quebrariam o binomial (k/n): são faixas que
# existem só no relatório de alterados (full outer da gold, ex. rótulos não informados).
n_antes = len(m)
m = m[m["total_exames_validos"].notna() & (m["total_exames_validos"] > 0)
      & m["quantidade_alterados"].notna()].copy()
print(f"Linhas usadas no modelo: {len(m)} (descartadas {n_antes - len(m)} com denominador NULL/0)")

def fit_binomial(X, k, n, iters=200, tol=1e-10):
    """Regressão logística ponderada sobre proporções (IRLS). Retorna beta e covariância."""
    beta = np.zeros(X.shape[1])
    for _ in range(iters):
        eta = np.clip(X @ beta, -30, 30)
        mu = 1 / (1 + np.exp(-eta))
        w = n * mu * (1 - mu) + 1e-9
        z = eta + (k / n - mu) / np.maximum(mu * (1 - mu), 1e-9)
        beta_new = np.linalg.solve(X.T @ (w[:, None] * X), X.T @ (w * z))
        if np.max(np.abs(beta_new - beta)) < tol:
            beta = beta_new
            break
        beta = beta_new
    eta = np.clip(X @ beta, -30, 30)
    mu = 1 / (1 + np.exp(-eta))
    w = n * mu * (1 - mu) + 1e-9
    cov = np.linalg.inv(X.T @ (w[:, None] * X))
    return beta, cov

idade_c = (m["idade"] - 47.5) / 10   # idade centralizada: OR por década a partir de 47,5 anos
ano_c = m["ano"] - 2022.5            # ano centralizado na série
X = np.column_stack([np.ones(len(m)), idade_c, ano_c])
beta, cov = fit_binomial(X, m["quantidade_alterados"].astype(float).values,
                          m["total_exames_validos"].astype(float).values)

se = np.sqrt(np.diag(cov))
names = ["intercepto (taxa aos 47,5 anos, ano central)", "idade (por década)", "ano (tendência)"]
print("Coeficientes (log-odds) e razão de chances (OR) com IC 95%:")
for nm, b, s in zip(names, beta, se):
    lo, hi = np.exp(b - 1.96 * s), np.exp(b + 1.96 * s)
    print(f"  {nm:<42} b={b:+.4f} (ep={s:.4f})   OR={np.exp(b):.3f}  IC95%[{lo:.3f}; {hi:.3f}]")

# Probabilidade prevista por faixa (padrão etário, no ano central 2022.5)
faixas_unicas = sorted(m["faixa_etaria"].unique(), key=lambda f: IDADE[f])
pred = pd.DataFrame({"faixa": faixas_unicas, "idade": [IDADE[f] for f in faixas_unicas]})
xx = np.column_stack([np.ones(len(pred)), (pred["idade"] - 47.5) / 10, np.zeros(len(pred))])
pred["p_prevista_pct"] = 100 / (1 + np.exp(-(xx @ beta)))
obs = taxa_faixa.toPandas().set_index("faixa_etaria")["taxa_pct"]
pred["p_observada_pct"] = pred["faixa"].map(obs)
print("\nProbabilidade prevista vs observada por faixa:")
print(pred.round(2).to_string(index=False))

In [0]:
# ---------------------------------------------------------------------------
# 8. PROJEÇÃO ILUSTRATIVA PARA 2026 (efeito ano do modelo, método delta)
# ---------------------------------------------------------------------------
for alvo in (2026,):
    x = np.array([1.0, 0.0, alvo - 2022.5])   # idade na média; efeito do ano
    p = 1 / (1 + np.exp(-(x @ beta)))
    var = float(x @ cov @ x)
    se_p = p * (1 - p) * np.sqrt(var)
    print(f"Taxa prevista para {alvo}: {p * 100:.2f}%  "
          f"(IC95% delta ≈ [{(p - 1.96 * se_p) * 100:.2f}%; {(p + 1.96 * se_p) * 100:.2f}%])")

print("\n⚠️ Extrapolação com apenas 6 pontos anuais e dados AGREGADOS (sem variáveis individuais).")
print("   Usar como indicador de TENDÊNCIA para planejamento, não como previsão clínica.")

In [0]:
# ---------------------------------------------------------------------------
# 9.C QUANTIDADE DE EXAMES ALTERADOS POR MUNICÍPIO × ANO
# ---------------------------------------------------------------------------
# A granularidade município×ano×faixa já existe na gold (quantidade_alterados);
# basta agregar. Conferência cruzada contra a silver (fonte original) garante
# que o numerador não foi distorcido pelo full outer da gold.
from pyspark.sql import functions as F

q5_muni_ano = (
    spark.table("`tcc-analise-siscan-10crs`.gold.fato_exames_alterados")
    .groupBy("municipio", "ano")
    .agg(F.sum("quantidade_alterados").alias("alterados_gold"))
)

silver_muni_ano = (
    spark.table("`tcc-analise-siscan-10crs`.silver.resultados_citopatologicos")
    .groupBy("municipio", "ano")
    .agg(F.sum("quantidade").alias("alterados_silver"))
)

div = (
    q5_muni_ano.alias("g")
    .join(silver_muni_ano.alias("s"), ["municipio", "ano"], how="full_outer")
    .select("municipio", "ano", F.col("g.alterados_gold"), F.col("s.alterados_silver"),
            (F.coalesce(F.col("g.alterados_gold"), F.lit(-1))
             - F.coalesce(F.col("s.alterados_silver"), F.lit(-2))).alias("diff"))
    .filter(F.col("diff") != 0)
)
print(f"Divergências gold × silver (município/ano): {div.count()}")

pivot_muni_ano = (
    q5_muni_ano.groupBy("municipio")
    .pivot("ano", [2020, 2021, 2022, 2023, 2024, 2025])
    .agg(F.sum("alterados_gold"))
    .toPandas()
    .set_index("municipio")
)
pivot_muni_ano["Total"] = pivot_muni_ano.sum(axis=1)
pivot_muni_ano.loc["Total"] = pivot_muni_ano.sum()
print("\nExames alterados por município e ano (10ª CRS, 2020–2025):")
print(pivot_muni_ano.fillna(0).astype(int).to_string())

In [0]:
# ---------------------------------------------------------------------------
# 9. MOTIVO DA ALTERAÇÃO — GRUPOS DE RESULTADO CITOPATOLÓGICO (silver)
# ---------------------------------------------------------------------------
# O relatório original (SISCAN/TABNET "resultados citopatológicos") traz a coluna
# grupo_resultado — verificar quais classificações existem antes de afirmar se é
# possível decompor o "alterado" em motivos (ASC-US, LSIL, HSIL, etc.).
from pyspark.sql import functions as F

res = spark.table("`tcc-analise-siscan-10crs`.silver.resultados_citopatologicos")

cat = (res.groupBy("grupo_resultado")
       .agg(F.count("*").alias("linhas"), F.sum("quantidade").alias("exames"))
       .orderBy(F.desc("exames")))
display(cat)

cob = (res.groupBy("ano")
       .agg(F.countDistinct("grupo_resultado").alias("categorias"), F.sum("quantidade").alias("exames"))
       .orderBy("ano"))
display(cob)

In [0]:
# ---------------------------------------------------------------------------
# 9.B VERIFICAÇÃO NA BRONZE — o motivo da alteração já estaria ingerido?
# ---------------------------------------------------------------------------
# Conferir se os CSVs brutos (bronze) trazem alguma dimensão de diagnóstico
# citológico (ASC-US, LSIL, HSIL, carcinoma...) que a silver tenha descartado.

raw = spark.table("`tcc-analise-siscan-10crs`.bronze.resultados_raw")
print("Schema bronze.resultados_raw:")
raw.printSchema()
print(f"Linhas: {raw.count()}")

print("\nColunas da bronze.resultados_raw (formato largo do TABNET — cada coluna é um valor da dimensão):")
print([c for c in raw.columns])

print("Arquivos-fonte ingeridos como 'resultados':")
raw.select("_file_source").distinct().orderBy("_file_source").show(truncate=False)

print("\nColunas de TODAS as tabelas bronze (busca de dimensão de diagnóstico):")
for t in ["adequabilidade_raw", "exames_geral_raw", "faixa_etaria_raw", "resultados_raw"]:
    print(f"\n--- {t}:")
    spark.table(f"`tcc-analise-siscan-10crs`.bronze.{t}").printSchema()

## 📌 Síntese — o que se pode extrair desta tabela

**Achados (reexecução de 03/nov/2026 — denominador = 130.598 exames analisáveis, excluindo 390 insatisfatórias e 70 rejeitadas):**
* Taxa global: **3,43%** (4.479 alterados / 130.598 exames analisáveis).
* Faixa com maior taxa: **10-14 anos (4,91%)** — porém base minúscula (8/163). Entre faixas robustas: **15-19 (4,40%)** e **35-39 (4,14%)**; na população-alvo, **25-29 (3,92%)** lidera.
* Padrão etário: a taxa **decresce com a idade** — modelo binomial: OR = **0,889 por década** de idade (IC95% 0,870–0,907), ou ~11% menos chance de resultado alterado a cada década.
* Evolução anual: 3,09% (2020) → pico **4,72% (2023)** → queda a **1,74% (2025)**; tendência −0,22 p.p./ano; efeito ano no modelo: OR = 0,927 (−7% ao ano).
* Grupos da diretriz: **<25 = 4,01%** > 25–64 = 3,43% > 65+ = 2,55% — o sobrerastreamento concentra a maior taxa, reforçando a diretriz de não rastrear <25 anos.
* Municípios: **Quaraí (4,60%)** e **Uruguaiana (4,55%)** no topo; **Sant'Ana do Livramento (1,69%)** e **São Gabriel (2,35%)** na base. IC de Wilson mostra que Maçambará (IC 1,88–6,62) e Santa Margarida (3,10–5,95) são estatisticamente imprecisos (poucos exames).
* Correlação volume × taxa (Spearman): **rho = −0,12** — não há relação entre volume de exames e taxa de alteração.
* Projeção ilustrativa 2026 (efeito ano): **2,53%** (IC95% 2,36–2,71).

### O que a tabela PERMITE
* Taxa de alteração ponderada por **faixa etária, ano e município** (razão de somas, com IC de Wilson para bases pequenas).
* **Padrão etário** do resultado alterado (via modelo logístico binomial: OR por década de idade).
* **Tendência temporal** 2020–2025 e projeção ilustrativa de taxa para 2026.
* **Ranking de municípios** com comparação estatisticamente honesta (IC 95%).

### O que a tabela NÃO permite
* **Motivo da alteração (verificado até a bronze — células 9 e 9-B):** `bronze.resultados_raw` está no formato largo original do TABNET com colunas apenas de **município + 15 faixas etárias + Total**; os arquivos-fonte são unicamente os 6 CSVs `q5-siscan_colo_alterados_faixa_etaria_*.csv` (2020–2025); e o volume de origem (`siscan_files_bronze/dataset`) contém apenas as 4 famílias de relatórios já ingeridas (Q1/Q2 exames geral, Q3 faixa etária, Q4 adequabilidade, Q5 alterados) — **nenhum relatório com dimensão de diagnóstico**. Não há quebra por ASC-US/LSIL/HSIL/carcinoma em nenhuma camada, bronze incluída. Para decompor o “alterado” seria necessário exportar do SISCAN/TABNET o relatório de resultados por diagnóstico e ingeri-lo no pipeline bronze→silver→gold.
* Distribuição etária **exata** das amostras insatisfatórias/rejeitadas — o SISCAN não cruza adequabilidade × faixa etária, então as exclusões do denominador são alocadas proporcionalmente ao volume de exames de cada faixa (pressuposto metodológico declarável).
* Prever resultado de um **exame individual** (dados agregados — falácia ecológica).
* Ajustar por fatores de confusão individuais (história reprodutiva, HPV, etc.).
* Diagnóstico de causa — alterado ≠ necessariamente lesão (inclui atipias benignas/ASC-US).
